In [12]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import string

## Dataset preparation
1. Load the email and keep only the relevant features
2. Load the Smishing dataset: since it lacks headers, we dynamically assign columns (v1,v2) and rename them for standardization
3.  Merge both datasets to build a more robust
4.  Function execution

In [13]:
def prepare_datasets():
    df_emails = pd.read_csv('spam_ham_dataset.csv')
    df_emails = df_emails[['label', 'text']].copy()
    df_emails.rename(columns={'text': 'original_text'}, inplace=True)
    df_emails['source'] = 'email'

    df_sms = pd.read_csv('spamhamdata.csv', sep='\t', header=None, names=['v1', 'v2'], encoding='latin-1')
    df_sms.rename(columns={'v1': 'label', 'v2': 'original_text'}, inplace=True)
    df_sms['source'] = 'sms'

    df_merged = pd.concat([df_emails, df_sms], ignore_index=True)
    return df_merged

In [14]:
df_base = prepare_datasets()
print(f"Total initial records: {len(df_base)}")

Total initial records: 10745


## Data Cleansing: Handling missing values and duplicates
1. We drop duplicate records based on the message content, keeping only the first occurrence. This is crucial to prevent model overfitting, ensuring the algorithm learns patterns rather than memorizing repeated spam compaigns
2. We remove rows where the original_text is empty (NaN)
3. Using the subset parameter ensures we only drop rows with nulls or duplicates in this critical text column, preserving the data point if non-essential metadata is missing
4. Execution of the cleaning function

In [15]:
def clean_duplicates_nulls(df):
    initial_count = len(df)

    df_clean = df.drop_duplicates(subset=['original_text'], keep='first').copy()
    df_clean = df_clean.dropna(subset=['original_text'])

    final_count = len(df_clean)
    print(f"Removed {initial_count - final_count} duplicate or null records.")
    return df_clean

In [16]:
df_unique = clean_duplicates_nulls(df_base)
print(f"Total records after cleaning: {len(df_unique)}")

Removed 5752 duplicate or null records.
Total records after cleaning: 4993


## Creating a checkpoint and structure verification
1. Local backup: We save the current progress to a CSV file (dataset_processed_phase1.csv). This acts as a reliable checkpoint; if the notebook kernel crashes or restarts in subsequent steps, we can load this file without rerunning the initial ingestion and cleaning processes
2. We use the index = False parameter to prevent Pandas from exporting the row indices as an unnamed "garbage" column
3. Verification: We print the dataset's structural information (info()) to confirm the absence of null values and check data types. Finally, we output a sample (head()) for a quick visual sanity check

In [17]:
df_unique.to_csv('data_processed_phase1.csv', index=False)
print("Processed data saved to 'data_processed_phase1.csv'\n")
print(df_unique.info())
print("\n ---First 5 records---")
print(df_unique.head())

Processed data saved to 'data_processed_phase1.csv'

<class 'pandas.DataFrame'>
Index: 4993 entries, 0 to 5170
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   label          4993 non-null   str   
 1   original_text  4993 non-null   object
 2   source         4993 non-null   str   
dtypes: object(1), str(2)
memory usage: 156.0+ KB
None

 ---First 5 records---
  label                                      original_text source
0   ham  Subject: enron methanol ; meter # : 988291\r\n...  email
1   ham  Subject: hpl nom for january 9 , 2001\r\n( see...  email
2   ham  Subject: neon retreat\r\nho ho ho , we ' re ar...  email
3  spam  Subject: photoshop , windows , office . cheap ...  email
4   ham  Subject: re : indian springs\r\nthis deal is t...  email


## Feature Engineering
Feature Engineering (Mathematical Extraction)
*   We extract mathematical metadata from the raw text before applying any Natural Language Processing (NLP) cleaning. This is a critical sequencing step, as subsequent text normalization will convert everything to lowercase and strip
numbers, permanently destroying these security indicators.
*   Message Length: Phishing attacks often feature longer texts to build persuasive social engineering narratives.
*   Uppercase Count: Detects a false "sense of urgency," a common psychological
tactic used by scammers (e.g., "UPDATE YOUR ACCOUNT NOW!").
*   Digit Count: Identifies the presence of obfuscated IP addresses, fraudulent phone numbers, or fake monetary traps.

In [18]:
def extract_mathematical_features(df):
  # Create a local copy to strictly avoid SettingWithCopyWarning
  df_features = df.copy()
  # 1. Total string length (Phishing attacks are often longer)
  df_features['message_length'] = df_features['original_text'].apply(lambda x:
  len(str(x)))
  # 2. Uppercase letter count (Indicator of Social Engineering / Sense of Urgency)
  df_features['num_uppercase'] = df_features['original_text'].apply(lambda x: sum(1
  for c in str(x) if c.isupper()))
  # 3. Digit count (Indicator of fake monetary prizes or obfuscated IPs)
  df_features['num_digits'] = df_features['original_text'].apply(lambda x: sum(1 for c in
  str(x) if c.isdigit()))
  return df_features
  # Function execution
  # We pass df_unique (the clean data without nulls/duplicates)
df_final = extract_mathematical_features(df_unique)

## Checkpoint: Post-Feature Extraction Backup  

*   Strategic Backup: We save a local checkpoint (dataset_processed_phase1_pre_nlp.csv) immediately after extracting the
mathematical metadata. This ensures that the computationally heavy calculations (length, uppercase, digits) are safely persisted before initiating resource-intensive NLP tasks.  
*   index=False is applied to prevent the creation of residual index columns
(Unnamed: 0).  
*   Quality Check: We output the .info() summary to verify that the newly
engineered numerical features contain no nulls, followed by .head() for a quick
visual validation of the enriched dataset.

In [19]:
# 1. Save a local CSV backup after mathematical feature extraction
# We name it 'pre_nlp' to indicate it's ready for linguistic cleaning
df_final.to_csv('dataset_processed_phase1_pre_nlp.csv', index=False)
# 2. Display the resulting structure to confirm the new numerical columns
print(df_final.info())
# 3. Print a visual sample to verify the extracted metadata
print("\n--- First 5 records view ---")
print(df_final.head())

<class 'pandas.DataFrame'>
Index: 4993 entries, 0 to 5170
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   label           4993 non-null   str   
 1   original_text   4993 non-null   object
 2   source          4993 non-null   str   
 3   message_length  4993 non-null   int64 
 4   num_uppercase   4993 non-null   int64 
 5   num_digits      4993 non-null   int64 
dtypes: int64(3), object(1), str(2)
memory usage: 273.1+ KB
None

--- First 5 records view ---
  label                                      original_text source  \
0   ham  Subject: enron methanol ; meter # : 988291\r\n...  email   
1   ham  Subject: hpl nom for january 9 , 2001\r\n( see...  email   
2   ham  Subject: neon retreat\r\nho ho ho , we ' re ar...  email   
3  spam  Subject: photoshop , windows , office . cheap ...  email   
4   ham  Subject: re : indian springs\r\nthis deal is t...  email   

   message_length  num_uppercase  num_digits  
0

## NLTK Setup and Memory Optimization
1. Resource Downloading: We download the essential packages from the NLTK library. This includes punkt (for word tokenization), stopwords (a dictionary of meaningless words), and additional corpora like wordnet (useful if we decide to upgrade from Stemming to Lemmatization later on)
2. Time Complexity Optimization: The stopwords are loaded into a set data structure rather than a traditional list. In Python, looking up an item in a list is O(N), whereas a set lookup is O(1). This architectural decision slashes the processing time for the entire dataset from minutes down to mere seconds
3. Initialization: We instantiate the PorterStemmer, the mathematical algorithm responsible for stripping word suffixes to extract their root form

In [20]:
#nltk.download('stopwords')
#nltk.download('punkt')
nltk.download(['punkt', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger', 'punkt_tab'])

stop_words_english = set(stopwords.words('english')) 
ps = PorterStemmer()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jonat\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jonat\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\jonat\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\jonat\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\jonat\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jonat\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


In [21]:
def transformation(text):
    text = text.lower()
    text = nltk.word_tokenize(text)
    
    y = [i for i in text if i.isalnum()]
    y = [i for i in y if i not in stop_words_english and i not in string.punctuation]
    y = [ps.stem(i) for i in y]
    
    return " ".join(y)

In [22]:

print("Initiating linguistic cleaning pipeline...")
df_final['clean_text'] = df_final['original_text'].apply(transformation)

print("---NLTK Cleaning completed successfully---")
df_final[['original_text', 'clean_text']].head()

Initiating linguistic cleaning pipeline...
---NLTK Cleaning completed successfully---


,original_text,clean_text
0,Subject: enron methanol ; meter # : 988291\r\n...,subject enron methanol meter 988291 follow not...
1,"Subject: hpl nom for january 9 , 2001\r\n( see...",subject hpl nom januari 9 2001 see attach file...
2,"Subject: neon retreat\r\nho ho ho , we ' re ar...",subject neon retreat ho ho ho around wonder ti...
3,"Subject: photoshop , windows , office . cheap ...",subject photoshop window offic cheap main tren...
4,Subject: re : indian springs\r\nthis deal is t...,subject indian spring deal book teco pvr reven...


# Feature Matrix and Preprocessing

## Objective

The objective of this phase is to transform the cleaned dataset into numerical structures that machine learning models can use for training.

This phase receives the cleaned DataFrame from Phase 1 and generates:

- `X`: the feature matrix.
- `Y`: the target vector.
- `preprocessor`: the transformation pipeline for text and numerical features.

---

## Inputs

The input for this phase is the cleaned DataFrame `df_final`.

The required columns are:

- `label`: original class of the message.
- `clean_text`: cleaned and processed SMS text.
- `message_length`: total number of characters in the original message.
- `num_uppercase`: number of uppercase letters in the original message.
- `num_digits`: number of numerical digits in the original message.

---

## Target Variable Encoding

The target column is converted from text labels into binary numerical values.

- `ham` becomes `0`
- `spam` becomes `1`

This creates the target vector `Y`.

In [23]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df_final["target"] = label_encoder.fit_transform(df_final["label"])

Y = df_final["target"]

print("Target encoding completed.")
print(df_final[["label", "target"]].drop_duplicates())

Target encoding completed.
  label  target
0   ham       0
3  spam       1


## Feature Selection

The input features are selected from the cleaned DataFrame.

The model will use both text-based and numerical features:

- `clean_text`: processed message text.
- `message_length`: message length.
- `num_uppercase`: uppercase character count.
- `num_digits`: digit count.

These columns create the feature matrix `X`.

In [24]:
X = df_final[["clean_text", "message_length", "num_uppercase", "num_digits"]]

print("Feature matrix created.")
print(X.head())

Feature matrix created.
                                          clean_text  message_length  \
0  subject enron methanol meter 988291 follow not...             327   
1  subject hpl nom januari 9 2001 see attach file...              97   
2  subject neon retreat ho ho ho around wonder ti...            2524   
3  subject photoshop window offic cheap main tren...             414   
4  subject indian spring deal book teco pvr reven...             336   

   num_uppercase  num_digits  
0              1          10  
1              1           9  
2              1          14  
3              1           0  
4              1           0  


## Text Vectorization and Numerical Scaling

The `ColumnTransformer` applies different transformations to different columns:

- `TfidfVectorizer` is applied to `clean_text` to convert text into numerical TF-IDF features.
- `MinMaxScaler` is applied to the numerical columns to scale their values between 0 and 1.

The final output of this preprocessor will be a combined numerical matrix containing both text and numerical features.

In [25]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "text_tfidf",
            TfidfVectorizer(),
            "clean_text"
        ),
        (
            "numeric_features",
            MinMaxScaler(),
            ["message_length", "num_uppercase", "num_digits"]
        )
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


## Outputs

At the end of this phase, the following objects are available:

- `X`: feature matrix with text and numerical columns.
- `Y`: binary target vector.
- `preprocessor`: transformation object that will convert raw features into numerical model-ready features.

These outputs will be used in Phase 3 for model training.

In [26]:
phase_2_outputs = {
    "X": X,
    "Y": Y,
    "preprocessor": preprocessor
}

print("Phase 2 completed successfully.")

Phase 2 completed successfully.


# Model Training

## Objective

The objective of this phase is to train multiple machine learning models using the feature matrix `X`, the target vector `Y`, and the `preprocessor` created in Phase 2.

This phase does not evaluate the models in detail. Evaluation will be performed in Phase 4.

The main goal of this phase is to generate:

- Trained models.
- Prediction vectors for the test set.
- The real test labels `Y_test`.

---

## Inputs

This phase receives the following objects from Phase 2:

- `X`: feature matrix.
- `Y`: target vector.
- `preprocessor`: transformation object for text and numerical features.

## Train/Test Split

The dataset is divided into training and testing sets.

The split uses:

- 80% of the data for training.
- 20% of the data for testing.
- `stratify=Y` to preserve the same proportion of spam and ham messages in both sets.

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    stratify=Y,
    random_state=42
)

print("Train/test split completed.")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Y_train shape:", Y_train.shape)
print("Y_test shape:", Y_test.shape)

Train/test split completed.
X_train shape: (3994, 4)
X_test shape: (999, 4)
Y_train shape: (3994,)
Y_test shape: (999,)


## Model Definition

Four machine learning models are defined:

1. Logistic Regression
2. Naive Bayes
3. Support Vector Machine
4. Random Forest

Models that support class weighting use `class_weight='balanced'` to reduce the impact of class imbalance.

In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ),

    "Naive Bayes": MultinomialNB(),

    "Support Vector Machine": LinearSVC(
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        class_weight="balanced_subsample",
        random_state=42
    )
}

print("Models defined successfully.")

Models defined successfully.


## Step 3: Pipeline Creation and Model Training

Each model is trained inside a `Pipeline`.

The pipeline performs two steps:

1. `preprocessor`: transforms text and numerical features.
2. `classifier`: trains the machine learning model.

Using a pipeline ensures that the same preprocessing steps are applied consistently during training and prediction.

In [29]:
from sklearn.pipeline import Pipeline

trained_models = {}

for model_name, model in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", model)
        ]
    )

    pipeline.fit(X_train, Y_train)
    trained_models[model_name] = pipeline

    print(f"{model_name} trained successfully.")

Logistic Regression trained successfully.
Naive Bayes trained successfully.
Support Vector Machine trained successfully.
Random Forest trained successfully.


## Step 4: Predictions on the Test Set

Each trained model is used to generate predictions on `X_test`.

These prediction vectors will be used in Phase 4 to evaluate and compare model performance.

In [30]:
y_pred_logistic = trained_models["Logistic Regression"].predict(X_test)
y_pred_nb = trained_models["Naive Bayes"].predict(X_test)
y_pred_svm = trained_models["Support Vector Machine"].predict(X_test)
y_pred_rf = trained_models["Random Forest"].predict(X_test)

print("Predictions generated successfully.")

Predictions generated successfully.


## Phase 3 Outputs

At the end of this phase, the following outputs are generated:

- `Y_test`: real labels from the test set.
- `y_pred_logistic`: predictions from Logistic Regression.
- `y_pred_nb`: predictions from Naive Bayes.
- `y_pred_svm`: predictions from Support Vector Machine.
- `y_pred_rf`: predictions from Random Forest.
- `trained_models`: dictionary containing the trained pipelines.

These outputs will be used in Phase 4 for model evaluation.

In [31]:
phase_3_outputs = {
    "Y_test": Y_test,
    "y_pred_logistic": y_pred_logistic,
    "y_pred_nb": y_pred_nb,
    "y_pred_svm": y_pred_svm,
    "y_pred_rf": y_pred_rf,
    "trained_models": trained_models
}

print("Phase 3 completed successfully.")

Phase 3 completed successfully.


In [ ]:
# ============================================================
# PHASE 4: MODEL EVALUATION
# ============================================================

# Required imports for evaluation and deployment
!pip install skl2onnx onnxruntime streamlit -q

import numpy as np
import json
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from skl2onnx import to_onnx

# ------------------------------------------------------------
# Phase 4 Objective:
# Evaluate the trained models generated in Phase 3.
#
# Required objects from Phase 3:
# - trained_pipelines
# - X_test
# - Y_test
# - X_train
# ------------------------------------------------------------

print("Initiating Zero Trust Evaluation...")

best_model_name = ""
best_pipeline = None
best_score = -1
metrics_log = "--- Model Evaluation Log ---\n\n"
trained_pipelines = phase_3_outputs["trained_models"]
for name, pipeline in trained_pipelines.items():
    y_pred = pipeline.predict(X_test)

    cm = confusion_matrix(Y_test, y_pred)
    precision = precision_score(Y_test, y_pred, zero_division=0)
    recall = recall_score(Y_test, y_pred, zero_division=0)

    log = (
        f"Model: {name}\n"
        f"Precision: {precision:.4f} | Recall: {recall:.4f}\n"
        f"Confusion Matrix:\n{cm}\n\n"
    )

    metrics_log += log
    print(log)

    # Selection formula:
    # Prioritize recall to detect spam,
    # but penalize false positives.
    score = recall - (cm[0][1] * 0.1)

    if score > best_score:
        best_score = score
        best_pipeline = pipeline
        best_model_name = name

print(f"\nWinning Model selected for Production: {best_model_name}")

# Save evaluation metrics
with open("metricas_entrenamiento.txt", "w") as f:
    f.write(metrics_log)

print("Evaluation metrics saved successfully.")


# ============================================================
# PHASE 5: MODEL EXPORT AND LOCAL DEPLOYMENT
# ============================================================

# ------------------------------------------------------------
# Phase 5 Objective:
# Export the best model selected in Phase 4 and generate
# a local Streamlit dashboard for inference.
# ------------------------------------------------------------

print("Transcoding to ONNX format...")

try:
    # Create a sample input row with the correct data types
    X_train_sample = X_train[:1].copy()

    X_train_sample["clean_text"] = X_train_sample["clean_text"].astype(str)
    X_train_sample["message_length"] = X_train_sample["message_length"].astype(np.float32)
    X_train_sample["num_uppercase"] = X_train_sample["num_uppercase"].astype(np.float32)
    X_train_sample["num_digits"] = X_train_sample["num_digits"].astype(np.float32)

    # Convert the best pipeline to ONNX
    onnx_model = to_onnx(best_pipeline, X_train_sample)

    # Save ONNX model
    with open("modelo_produccion.onnx", "wb") as f:
        f.write(onnx_model.SerializeToString())

    print("ONNX model exported successfully: modelo_produccion.onnx")

except Exception as e:
    print(f"Error during ONNX conversion: {e}")


# ------------------------------------------------------------
# Version control file
# ------------------------------------------------------------

try:
    with open("version.json", "r") as f:
        current_version = json.load(f).get("version", 0)
except FileNotFoundError:
    current_version = 0

with open("version.json", "w") as f:
    json.dump({"version": current_version + 1}, f)

print(f"version.json updated to version {current_version + 1}")


# ------------------------------------------------------------
# Streamlit dashboard generator
# ------------------------------------------------------------

app_code = """import streamlit as st
import onnxruntime as rt
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import string

st.set_page_config(page_title="Spam ML Dashboard", layout="wide")

st.title("Anti-Spam Filter - Local ML Inference")
st.markdown("The ONNX model is running directly on your local machine.")

@st.cache_resource
def setup_nlp():
    nltk.download("punkt", quiet=True)
    nltk.download("stopwords", quiet=True)
    return PorterStemmer(), set(stopwords.words("english"))

ps, stop_words_english = setup_nlp()

def clean_input(text):
    text = text.lower()
    text = nltk.word_tokenize(text)
    y = [i for i in text if i.isalnum()]
    y = [i for i in y if i not in stop_words_english and i not in string.punctuation]
    y = [ps.stem(i) for i in y]
    return " ".join(y)

@st.cache_resource
def load_model():
    return rt.InferenceSession("modelo_produccion.onnx")

try:
    sess = load_model()
    model_loaded = True
except Exception as e:
    st.error(f"Error loading ONNX model: {e}")
    model_loaded = False

col1, col2 = st.columns([2, 1])

with col1:
    st.subheader("SMS / Email Evaluator")

    user_input = st.text_area(
        "Paste the message here:",
        height=150
    )

    if st.button("Evaluate"):
        if user_input and model_loaded:
            message_length = np.float32(len(str(user_input)))
            num_uppercase = np.float32(sum(1 for c in str(user_input) if c.isupper()))
            num_digits = np.float32(sum(1 for c in str(user_input) if c.isdigit()))

            cleaned_text = clean_input(user_input)

            inputs = {
                "clean_text": np.array([[cleaned_text]]),
                "message_length": np.array([[message_length]], dtype=np.float32),
                "num_uppercase": np.array([[num_uppercase]], dtype=np.float32),
                "num_digits": np.array([[num_digits]], dtype=np.float32)
            }

            label_name = sess.get_outputs()[0].name
            prediction = sess.run([label_name], inputs)[0]

            st.markdown("---")

            if prediction[0] == 1:
                st.error("SPAM DETECTED")
            else:
                st.success("LEGITIMATE MESSAGE / HAM")

            st.caption(
                f"Length: {int(message_length)} | "
                f"Uppercase: {int(num_uppercase)} | "
                f"Digits: {int(num_digits)}"
            )

        else:
            st.warning("Please enter a valid message.")

with col2:
    st.subheader("Evaluation Logs")

    try:
        with open("metricas_entrenamiento.txt", "r") as f:
            st.code(f.read(), language="text")
    except:
        st.write("Logs not available.")
"""

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("Dashboard code generated successfully: app.py")

## Zero Trust Model Evaluation and Edge Deployment
1. **Model Auditing:** Evaluate trained models using Precision, Recall, and the Confusion Matrix to ensure reliability.
2. **Model Selection:** Select the optimal algorithm that maximizes spam detection (Recall) while strictly minimizing false positives.
3. **ONNX Transcoding:** Convert the winning pipeline into an `.onnx` binary. This removes Python dependencies, allowing the ML process to happen directly on the local machine via Edge Inference.
4. **Cache Control:** Update `version.json` to signal the frontend about the new model version.
5. **Local Dashboard Generation:** Generate a standalone Streamlit application (`app.py`) equipped with database connectivity for local execution.

In [33]:
# Instalación de librerías estrictamente necesarias para esta sección
!pip install skl2onnx onnxruntime streamlit psycopg2-binary -q

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score
import json
from skl2onnx import to_onnx

# --- Preparación previa (Crucial para tener modelos que evaluar) ---
# Codificamos la variable objetivo: 'ham' -> 0, 'spam' -> 1
le = LabelEncoder()
df_final['target'] = le.fit_transform(df_final['label']) 

X = df_final[['clean_text', 'message_length', 'num_uppercase', 'num_digits']]
Y = df_final['target']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=42)

# Unificamos el procesamiento de texto (TF-IDF) y números (MinMaxScaler) en un solo pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(), 'clean_text'),
        ('num', MinMaxScaler(), ['message_length', 'num_uppercase', 'num_digits'])
    ])

models = {
    'LogisticRegression': LogisticRegression(class_weight='balanced', max_iter=1000),
    'NaiveBayes': MultinomialNB(), 
    'SVM': SVC(kernel='linear', class_weight='balanced', probability=True),
    'RandomForest': RandomForestClassifier(class_weight='balanced_subsample', random_state=42)
}

# Entrenamos los modelos en memoria rápidamente
trained_pipelines = {}
for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    pipeline.fit(X_train, Y_train)
    trained_pipelines[name] = pipeline

# --- Inicio de la Evaluación "Zero Trust" ---
print("Initiating Zero Trust Evaluation...")
best_model_name = ""
best_pipeline = None
best_score = -1
metrics_log = "--- Model Evaluation Log ---\n\n"

for name, pipeline in trained_pipelines.items():
    y_pred = pipeline.predict(X_test)
    cm = confusion_matrix(Y_test, y_pred)
    precision = precision_score(Y_test, y_pred, zero_division=0)
    recall = recall_score(Y_test, y_pred, zero_division=0)
    
    log = f"Model: {name}\nPrecision: {precision:.4f} | Recall: {recall:.4f}\nConfusion Matrix:\n{cm}\n\n"
    metrics_log += log
    print(log)
    
    # CRUCIAL: Fórmula de selección. Buscamos el mayor Recall (detectar spam) 
    # pero restamos puntos si el modelo tiene Falsos Positivos (cm[0][1])
    score = recall - (cm[0][1] * 0.1)
    if score > best_score:
        best_score = score
        best_pipeline = pipeline
        best_model_name = name

print(f"\nWinning Model selected for Production: {best_model_name}")

with open("metricas_entrenamiento.txt", "w") as f:
    f.write(metrics_log)

# --- Transcodificación a ONNX ---
print("Transcoding to ONNX format...")
try:
    # Preparamos un ejemplo simulado con los tipos de datos exactos para configurar el modelo ONNX
    X_train_sample = X_train[:1].copy()
    X_train_sample['clean_text'] = X_train_sample['clean_text'].astype(str)
    X_train_sample['message_length'] = X_train_sample['message_length'].astype(np.float32)
    X_train_sample['num_uppercase'] = X_train_sample['num_uppercase'].astype(np.float32)
    X_train_sample['num_digits'] = X_train_sample['num_digits'].astype(np.float32)

    onnx_model = to_onnx(best_pipeline, X_train_sample)
    
    # Exportamos el modelo matemáticamente puro para usarlo en local
    with open("modelo_produccion.onnx", "wb") as f:
        f.write(onnx_model.SerializeToString())
    print("ONNX model exported successfully: modelo_produccion.onnx")
except Exception as e:
    print(f"Error during ONNX conversion: {e}")

# --- Control de Caché ---
try:
    with open("version.json", "r") as f:
        current_version = json.load(f).get("version", 0)
except FileNotFoundError:
    current_version = 0

with open("version.json", "w") as f:
    json.dump({"version": current_version + 1}, f)
print(f"version.json updated to version {current_version + 1}\n")

# --- Generador del Dashboard Local ---
app_code = """import streamlit as st
import onnxruntime as rt
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import string
import psycopg2

# Credenciales de base de datos extraídas del ticket
DB_USER = "aegis"
DB_PASS = "LnaXQfQpN2rXHwYl"

st.set_page_config(page_title="Spam ML Dashboard", layout="wide")
st.title("🛡️ Anti-Spam Filter (Local ML Inference)")
st.markdown("The ONNX model is running directly on your local machine.")

@st.cache_resource
def setup_nlp():
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    return PorterStemmer(), set(stopwords.words('english'))

ps, stop_words_english = setup_nlp()

def clean_input(text):
    text = text.lower()
    text = nltk.word_tokenize(text)
    y = [i for i in text if i.isalnum()]
    y = [i for i in y if i not in stop_words_english and i not in string.punctuation]
    y = [ps.stem(i) for i in y]
    return " ".join(y)

@st.cache_resource
def load_model():
    return rt.InferenceSession("modelo_produccion.onnx")

try:
    sess = load_model()
    model_loaded = True
except Exception as e:
    st.error(f"Error loading ONNX: {e}")
    model_loaded = False

col1, col2 = st.columns([2, 1])

with col1:
    st.subheader("SMS/Email Evaluator")
    user_input = st.text_area("Paste the message here:", height=150)
    
    if st.button("Evaluate"):
        if user_input and model_loaded:
            # Extracción de características estáticas
            m_len = np.float32(len(str(user_input)))
            num_up = np.float32(sum(1 for c in str(user_input) if c.isupper()))
            num_dig = np.float32(sum(1 for c in str(user_input) if c.isdigit()))
            
            cleaned_text = clean_input(user_input)
            
            # Formateo de las variables como lo espera el modelo ONNX local
            inputs = {
                'clean_text': np.array([[cleaned_text]]),
                'message_length': np.array([[m_len]], dtype=np.float32),
                'num_uppercase': np.array([[num_up]], dtype=np.float32),
                'num_digits': np.array([[num_dig]], dtype=np.float32)
            }
            
            label_name = sess.get_outputs()[0].name
            pred = sess.run([label_name], inputs)[0]
            
            st.markdown("---")
            if pred[0] == 1:
                st.error("🚨 **SPAM DETECTED**")
            else:
                st.success("✅ **LEGITIMATE MESSAGE (HAM)**")
            
            st.caption(f"Length: {int(m_len)} | Uppercase: {int(num_up)} | Digits: {int(num_dig)}")
        else:
            st.warning("Please enter a valid text.")

with col2:
    st.subheader("Evaluation Logs")
    try:
        with open("metricas_entrenamiento.txt", "r") as f:
            st.code(f.read(), language="text")
    except:
        st.write("Logs not available.")
"""

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)
print("Dashboard code generated successfully: app.py")


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Initiating Zero Trust Evaluation...
Model: LogisticRegression
Precision: 0.8563 | Recall: 0.9966
Confusion Matrix:
[[657  49]
 [  1 292]]


Model: NaiveBayes
Precision: 0.9863 | Recall: 0.7372
Confusion Matrix:
[[703   3]
 [ 77 216]]


Model: SVM
Precision: 0.9541 | Recall: 0.9932
Confusion Matrix:
[[692  14]
 [  2 291]]


Model: RandomForest
Precision: 0.9241 | Recall: 0.9556
Confusion Matrix:
[[683  23]
 [ 13 280]]



Winning Model selected for Production: NaiveBayes
Transcoding to ONNX format...
ONNX model exported successfully: modelo_produccion.onnx
version.json updated to version 2

Dashboard code generated successfully: app.py
